# Test des paramètres de la forêt aléatoire

Auteur : Zohair

On refait la préparation du notebook `02_modelisation`, puis on change seulement le `random_state` de la forêt aléatoire (15, 42, 100) pour voir si les résultats bougent.

## 1. Chargement et préparation (mêmes étapes que le notebook 02)

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn import metrics

df = pd.read_csv("../data/hotel_bookings_clean.csv")

num_cols = ["lead_time", "stays_in_weekend_nights", "stays_in_week_nights", "adults", "children", "babies",
            "is_repeated_guest", "previous_cancellations", "previous_bookings_not_canceled", "adr"]
cat_cols = ["hotel", "arrival_date_month", "meal", "market_segment", "distribution_channel",
            "reserved_room_type", "deposit_type", "customer_type"]

X = pd.get_dummies(df[num_cols + cat_cols], columns=cat_cols, dtype=int)
y = df["is_canceled"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler_std = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[num_cols] = scaler_std.fit_transform(X_train[num_cols])
X_test_scaled[num_cols] = scaler_std.transform(X_test[num_cols])
print(X_train_scaled.shape, X_test_scaled.shape)

(69766, 55) (17442, 55)


## 2. On change seulement le random_state de la forêt

Le découpage train / test reste le même (`random_state=42`). Pour chaque valeur, on affiche l'AUC sur le test, puis la règle des 100 : nombre d'annulations parmi les 100 réservations les plus risquées sur 1 000, avec et sans les Non Refund.

In [2]:
for valeur in [15, 42, 100]:
    foret = RandomForestClassifier(n_estimators=100, max_depth=20, random_state=valeur)
    foret.fit(X_train_scaled, y_train)
    test = df.loc[X_test.index, ["deposit_type"]].copy()
    test["proba"] = foret.predict_proba(X_test_scaled)[:, 1]
    test["annule"] = y_test
    lot = test.sample(1000, random_state=42)
    top100 = lot.nlargest(100, "proba")
    top100_sans_nr = lot[lot["deposit_type"] != "Non Refund"].nlargest(100, "proba")
    print("random_state =", valeur,
          "| AUC :", round(metrics.roc_auc_score(y_test, test["proba"]), 4),
          "| top 100 :", top100["annule"].sum(),
          "| top 100 sans Non Refund :", top100_sans_nr["annule"].sum())

random_state = 15 | AUC : 0.7968 | top 100 : 77 | top 100 sans Non Refund : 72
random_state = 42 | AUC : 0.7963 | top 100 : 75 | top 100 sans Non Refund : 68
random_state = 100 | AUC : 0.7962 | top 100 : 77 | top 100 sans Non Refund : 70


**Observations**

- L'AUC reste presque identique : 0,7968 avec 15, 0,7963 avec 42, 0,7962 avec 100.
- La règle des 100 bouge un peu : de 75 à 77 annulations sur 100, et de 68 à 72 sans les Non Refund.
- Le résultat ne dépend donc pas d'un tirage chanceux : les chiffres changent légèrement, mais la conclusion reste la même. On garde 42 dans les notebooks rendus pour que tout le monde retrouve les mêmes chiffres.

## 3. Optimisation complète avec GridSearchCV

On teste plusieurs réglages ensemble, en validation croisée sur l'entraînement seulement (le test n'est pas utilisé pour choisir) :
- `max_depth` : profondeur maximale des arbres ;
- `n_estimators` : nombre d'arbres ;
- `min_samples_leaf` : nombre minimum de réservations dans une feuille.

In [3]:
from sklearn.model_selection import GridSearchCV

grille = {
    "max_depth": [15, 20, 25],
    "n_estimators": [100, 200],
    "min_samples_leaf": [1, 5],
}
recherche = GridSearchCV(RandomForestClassifier(random_state=42), grille, cv=3, scoring="roc_auc", n_jobs=-1)
recherche.fit(X_train_scaled, y_train)

print("Meilleurs réglages :", recherche.best_params_)
print("AUC en validation croisée :", round(recherche.best_score_, 4))

Meilleurs réglages : {'max_depth': 20, 'min_samples_leaf': 1, 'n_estimators': 200}
AUC en validation croisée : 0.7881


In [4]:
resultats = pd.DataFrame(recherche.cv_results_)
resultats = resultats[["param_max_depth", "param_n_estimators", "param_min_samples_leaf", "mean_test_score"]]
resultats.sort_values("mean_test_score", ascending=False).round(4)

,param_max_depth,param_n_estimators,param_min_samples_leaf,mean_test_score
5,20,200,1,0.7881
4,20,100,1,0.7869
9,25,200,1,0.7860
11,25,200,5,0.7859
10,25,100,5,0.7853
7,20,200,5,0.7842
8,25,100,1,0.7841
6,20,100,5,0.7837
1,15,200,1,0.7826
0,15,100,1,0.7818


## 4. Le meilleur modèle sur le test et la règle des 100

In [5]:
meilleur = recherche.best_estimator_
test = df.loc[X_test.index, ["deposit_type"]].copy()
test["proba"] = meilleur.predict_proba(X_test_scaled)[:, 1]
test["annule"] = y_test
lot = test.sample(1000, random_state=42)

print("AUC test :", round(metrics.roc_auc_score(y_test, test["proba"]), 4))
print("Top 100 :", lot.nlargest(100, "proba")["annule"].sum())
print("Top 100 sans Non Refund :", lot[lot["deposit_type"] != "Non Refund"].nlargest(100, "proba")["annule"].sum())

AUC test : 0.7969
Top 100 : 76
Top 100 sans Non Refund : 69


**Observations**

- La recherche teste 12 combinaisons. La meilleure est max_depth = 20, n_estimators = 200, min_samples_leaf = 1, avec une AUC de 0,7881 en validation croisée.
- Notre modèle actuel (max_depth = 20, n_estimators = 100, min_samples_leaf = 1) arrive juste derrière, avec 0,7869.
- Sur le test, le meilleur modèle obtient une AUC de 0,7969, contre 0,7963 pour notre modèle. Pour la règle des 100, il trouve 76 annulations (69 sans les Non Refund), contre 75 et 68.
- L'optimisation complète confirme notre choix : la profondeur de 20 est la bonne, et doubler le nombre d'arbres n'apporte presque rien. On garde le modèle actuel, plus rapide à entraîner.